In [ ]:
import sys
import os
import subprocess as sp
from io import StringIO
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
CSV_DIR = "csv"
os.makedirs(CSV_DIR, exist_ok=True)

def get_csv_path(file_name):
    csv_name = f"{file_name}.csv" if not file_name.endswith('.csv') else file_name
    return os.path.join(CSV_DIR, csv_name)

def read_sql_file(file_name):
    sql_name = f"{file_name}.sql"
    with open(sql_name, 'r') as f:
        return f.read()

def fetch_data(sql_query):
    print(f"Fetching data from {sql_query}")
    try:
        data = pd.read_csv(
            StringIO(
                sp.check_output(
                    f"""pharos sql run --sql "{sql_query}" | jq -r '.result.data'""",
                    shell=True,
                ).decode("utf-8")
            )
        )
    except Exception as e:
        print(f"Error executing SQL query: {e}")
        sys.exit(1)
    return data

def save_to_csv(df, file_name):
    csv_path = get_csv_path(file_name)
    df.to_csv(csv_path, index=False)
    print(f"Saved to {csv_path}")
    return csv_path

def select_all_from_table(table_name, use_cache=True):
    file_name = table_name.split('.')[-1]
    csv_path = get_csv_path(file_name)

    if use_cache and os.path.exists(csv_path):
        print(f"Loading {csv_path} from cache...")
        data = pd.read_csv(csv_path)
    else:
        if "swh.migration_event_log" in table_name:
            print("Note: This table is large; fetching will timeout.")
            return None
        print(f"Fetching data from table: {table_name}...")
        sql_query = f"SELECT * FROM {table_name}"
        if "swh" or "dataload_metrics_deployment_data" in table_name:
            sql_query += " WHERE DATE(wd_event_date) <= CURRENT_DATE"
        sql_query += " LIMIT 21052"
        data = fetch_data(sql_query)
        if use_cache:
            data.to_csv(csv_path, index=False)
            print(f"Saved to {csv_path}")

    return data

In [ ]:
tables = {}

In [ ]:
table_names = [
    "lookup_db.sfdc_account_details",
    "lookup_db.sfdc_customer_tenants",
    "lookup_db.sfdc_customer_account_tenants",
    "lookup_db.sfdc_deployments",
    "lookup_db.sfdc_account_tenant_map",
    "swh.change_tracker_event_log",
    "swh.tenant_compare_event_log",
    "swh.scopes_metrics",
    "swh.scopes_input_type_metrics",
    "swh.migration_event_log",
    "swh.tenant_build",
    "swh.dataload_metrics",
    "cdt.implementation_type_mapping",
    "cdt.dataload_metrics_deployment_data",
    "cdt.migrations_blended",
]

for table_name in table_names:
    tables[table_name] = select_all_from_table(table_name)
    if tables[table_name] is not None:
        print(f"Loaded {table_name}: {len(tables[table_name])} rows, {len(tables[table_name].columns)} columns")

In [ ]:
files = []

In [ ]:
for table_name, df in tables.items():
    if df is None:
        continue
    print(f"\n{table_name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  First few rows:\n{df.head()}")

In [ ]:
# Replicate testing.sql logic in Python
def get_qualifying_tenant_count(tables_dict, months_back=6):
    """
    Replicates the logic from testing.sql:
    Count distinct qualifying tenants that have:
    - Migration events in last 6 months
    - Scope metrics with input type in last 6 months
    - Active deployments
    - Tenant type in ('PROD', 'SANDBOX', 'SANDBOX-PREVIEW')
    """
    mel = tables_dict['swh.migration_event_log'].copy()
    sm = tables_dict['swh.scopes_metrics'].copy()
    sit = tables_dict['swh.scopes_input_type_metrics'].copy()
    map_df = tables_dict['lookup_db.sfdc_account_tenant_map'].copy()
    deployments = tables_dict['lookup_db.sfdc_deployments'].copy()
    
    # Calculate cutoff date (6 months back)
    cutoff_date = datetime.now() - timedelta(days=months_back * 30)
    
    # Convert date columns to datetime and filter by date
    if 'wd_event_date' in mel.columns:
        mel['wd_event_date'] = pd.to_datetime(mel['wd_event_date'], errors='coerce')
    mel_filtered = mel[mel['wd_event_date'] >= cutoff_date].copy()
    
    if 'wd_event_date' in sm.columns:
        sm['wd_event_date'] = pd.to_datetime(sm['wd_event_date'], errors='coerce')
    sm_filtered = sm[sm['wd_event_date'] >= cutoff_date].copy()
    
    if 'wd_event_date' in sit.columns:
        sit['wd_event_date'] = pd.to_datetime(sit['wd_event_date'], errors='coerce')
    sit_filtered = sit[
        (sit['wd_event_date'] >= cutoff_date) &
        (sit['input_type'].notna())
    ].copy()
    
    # Step 1: Find migration tenants (CTE equivalent)
    # Join migration_event_log with scopes_metrics
    mel_sm = mel_filtered.merge(
        sm_filtered,
        left_on='source_object_id',
        right_on='scope_external_id',
        how='left',
        suffixes=('', '_sm')
    )
    
    # Join with scopes_input_type_metrics
    mel_sm_sit = mel_sm.merge(
        sit_filtered,
        left_on='source_object_id',
        right_on='scope_external_id',
        how='left',
        suffixes=('', '_sit')
    )
    
    # Filter: input_type not null and tenant_name not null
    migration_tenants = mel_sm_sit[
        (mel_sm_sit['input_type'].notna()) &
        (mel_sm_sit['tenant_name'].notna())
    ]['tenant_name'].unique()
    
    # Step 2: Join with account mapping and details
    migration_tenants_df = pd.DataFrame({'tenant_name': migration_tenants})
    
    # Join with sfdc_account_tenant_map
    result = migration_tenants_df.merge(
        map_df,
        on='tenant_name',
        how='inner'
    )
    
    # Join with sfdc_deployments
    result = result.merge(
        deployments,
        left_on='sf_account_id',
        right_on='customer_sf_account_id',
        how='inner'
    )
    
    # Filter by overall_status and tenant_type
    result_filtered = result[
        (result['overall_status'] == 'Active') &
        (result['tenant_type'].isin(['PROD', 'SANDBOX', 'SANDBOX-PREVIEW']))
    ]
    
    # Count distinct tenants
    qualifying_count = result_filtered['tenant_name'].nunique()
    
    return qualifying_count

# Run the query
qualifying_count = get_qualifying_tenant_count(tables)
print(f"\nQualifying tenant count: {qualifying_count}")

In [ ]:
# Replicate testing2.sql logic in Python
def get_qualifying_tenants_details(tables_dict, months_back=6):
    """
    Replicates the logic from testing2.sql:
    Get account_name, tenant_type, and tenant_name for qualifying tenants
    """
    mel = tables_dict['swh.migration_event_log'].copy()
    sm = tables_dict['swh.scopes_metrics'].copy()
    sit = tables_dict['swh.scopes_input_type_metrics'].copy()
    map_df = tables_dict['lookup_db.sfdc_account_tenant_map'].copy()
    deployments = tables_dict['lookup_db.sfdc_deployments'].copy()
    account_details = tables_dict['lookup_db.sfdc_account_details'].copy()
    
    # Calculate cutoff date (6 months back)
    cutoff_date = datetime.now() - timedelta(days=months_back * 30)
    
    # Convert date columns to datetime and filter by date
    if 'wd_event_date' in mel.columns:
        mel['wd_event_date'] = pd.to_datetime(mel['wd_event_date'], errors='coerce')
    mel_filtered = mel[mel['wd_event_date'] >= cutoff_date].copy()
    
    if 'wd_event_date' in sm.columns:
        sm['wd_event_date'] = pd.to_datetime(sm['wd_event_date'], errors='coerce')
    sm_filtered = sm[sm['wd_event_date'] >= cutoff_date].copy()
    
    if 'wd_event_date' in sit.columns:
        sit['wd_event_date'] = pd.to_datetime(sit['wd_event_date'], errors='coerce')
    sit_filtered = sit[
        (sit['wd_event_date'] >= cutoff_date) &
        (sit['input_type'].notna())
    ].copy()
    
    # Step 1: Find migration tenants (same as testing.sql)
    mel_sm = mel_filtered.merge(
        sm_filtered,
        left_on='source_object_id',
        right_on='scope_external_id',
        how='left',
        suffixes=('', '_sm')
    )
    
    mel_sm_sit = mel_sm.merge(
        sit_filtered,
        left_on='source_object_id',
        right_on='scope_external_id',
        how='left',
        suffixes=('', '_sit')
    )
    
    migration_tenants = mel_sm_sit[
        (mel_sm_sit['input_type'].notna()) &
        (mel_sm_sit['tenant_name'].notna())
    ]['tenant_name'].unique()
    
    # Step 2: Join with account mapping and deployments
    migration_tenants_df = pd.DataFrame({'tenant_name': migration_tenants})
    
    result = migration_tenants_df.merge(
        map_df,
        on='tenant_name',
        how='inner'
    )
    
    # Join with deployments
    result = result.merge(
        deployments,
        left_on='sf_account_id',
        right_on='customer_sf_account_id',
        how='inner'
    )
    
    # Join with account_details to get account_name
    result = result.merge(
        account_details,
        left_on='sf_account_id',
        right_on='sf_account_id',
        how='inner'
    )
    
    # Filter by overall_status and tenant_type
    result_filtered = result[
        (result['overall_status'] == 'Active') &
        (result['tenant_type'].isin(['PROD', 'SANDBOX', 'SANDBOX-PREVIEW']))
    ]
    
    # Select and return the required columns, sorted
    final_result = result_filtered[['account_name', 'tenant_type', 'tenant_name']].drop_duplicates()
    final_result = final_result.sort_values('account_name')
    
    return final_result

# Run the query
qualifying_tenants = get_qualifying_tenants_details(tables)
print(f"\nQualifying tenants details ({len(qualifying_tenants)} rows):")
print(qualifying_tenants)

In [ ]:
# Access individual tables easily by name
# Examples:
# tables['swh.scopes_metrics']
# tables['swh.migration_event_log']
# tables['lookup_db.sfdc_account_details']

# You can also save results to CSV if needed
# qualifying_tenants.to_csv('csv/qualifying_tenants.csv', index=False)